In [0]:
# Databricks notebook source
# /// script
# [tool.databricks.environment]
# environment_version = "5"
# ///

# MAGIC %md
# MAGIC # Mapping Execution: Target Model 기반 실행
# MAGIC
# MAGIC 현재 개별 테스트를 위해 `COMPLAINT` Target만 실행한다.
# MAGIC Target 유형(DIRECT/ENTITY/MASTER) 판정과
# MAGIC 소스 조회, `integrate()` / `load_master_data()` 호출은
# MAGIC `mapping_orchestrator.run_target()`이 담당한다.

# COMMAND ----------

# MAGIC %md
# MAGIC ## 0. Job Run ID 수신
# MAGIC
# MAGIC Job 설정의 base_parameters(`job_run_id`: `{{job.run_id}}`)로 전달된 값을 읽는다.
# MAGIC 대화형 실행에서는 빈 값이 들어오며 None으로 처리한다.

# COMMAND ----------

dbutils.widgets.text("job_run_id", "")

job_run_id = dbutils.widgets.get("job_run_id") or None

# COMMAND ----------

import os
import sys

for _p in sys.path:
    _parent = os.path.dirname(_p)
    if os.path.isdir(os.path.join(_parent, "src")) and _parent not in sys.path:
        sys.path.insert(0, _parent)
        break

try:
    import src.mapping.mapping_config as cfg
    import src.mapping.mapping_engine as engine_mod
    import src.mapping.mapping_orchestrator as orch
except ModuleNotFoundError:
    import mapping_config as cfg
    import mapping_engine as engine_mod
    import mapping_orchestrator as orch

engine = engine_mod.MappingEngine.from_tables(spark)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. 실행 대상
# MAGIC
# MAGIC 현재 개별 E2E 테스트를 위해 COMPLAINT만 실행한다.

# COMMAND ----------

TARGET_TABLE = "COMPLAINT"

targets = [TARGET_TABLE]

print(f"🎯 Mapping Execution 대상: {TARGET_TABLE}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Target 실행

# COMMAND ----------

results = {}

for target_table in targets:
    print(
        f"\n{'=' * 60}\n"
        f"[{target_table}] Mapping Execution 시작\n"
        f"{'=' * 60}"
    )

    result = orch.run_target(
        engine,
        target_table,
        job_run_id=job_run_id,
    )

    results[target_table] = result

    print(f"integration_type = {result['integration_type']}")
    print(f"sources          = {result['sources']}")

    if "integration_result" in result:
        print(f"integrate 결과   = {result['integration_result']}")

    if "master_load_result" in result:
        print(f"master 적재 결과 = {result['master_load_result']}")

    if "note" in result:
        print(f"비고             = {result['note']}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. 실행 결과 요약

# COMMAND ----------

print("Mapping Execution 완료:", list(results.keys()))

for target_table, result in results.items():
    n_rows = None
    table = cfg.gold_candidate_table(target_table)

    if spark.catalog.tableExists(table):
        n_rows = spark.table(table).count()

    print(
        f"  {target_table:10s} "
        f"integration_type={result['integration_type']:8s} "
        f"gold_candidate 행수={n_rows}"
    )